<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 22**
---

# **Parameter-Efficient Fine-Tuning (PEFT)**

En el notebook anterior vimos que adaptar un modelo a un dominio consiste en seguir entrenándolo con datos de ese dominio. El **ajuste fino completo** (*full fine-tuning*) actualiza todos los parámetros del modelo, y eso plantea tres problemas a medida que los modelos crecen:

- **Cómputo y memoria.** Además de los pesos, durante el entrenamiento hay que guardar el gradiente de cada parámetro y los estados del optimizador (Adam mantiene dos valores adicionales por parámetro). Un modelo que cabe en una GPU para inferencia puede no caber para entrenarlo.
- **Almacenamiento.** Cada tarea produce una copia completa del modelo. Un modelo de 7.000 millones de parámetros ocupa unos 14 GB en 16 bits; diez tareas son diez copias.
- **Sobreajuste y olvido catastrófico.** Con pocos datos y miles de millones de parámetros libres, el modelo puede memorizar los ejemplos y perder capacidades generales que tenía.

Las técnicas **PEFT** (*Parameter-Efficient Fine-Tuning*) parten de una idea común: **congelar los pesos preentrenados $\theta_0$ y entrenar únicamente un conjunto pequeño de parámetros $\phi$**, nuevos o seleccionados, que normalmente supone menos del 1 % del total:

$$
\theta_0 \ \text{(congelado)} \;+\; \phi \ \text{(entrenable)}, \qquad |\phi| \ll |\theta_0|
$$

Lo sorprendente es que, en la mayoría de las tareas, el resultado es comparable al del ajuste fino completo. Antes de repasar los métodos, merece la pena entender por qué.

## **1. ¿Por qué funcionan? La intuición**

### 1.1. El modelo preentrenado ya hace casi todo el trabajo

Un modelo preentrenado ya dispone de representaciones internas muy ricas: sintaxis, significado, conocimiento del mundo. Adaptarlo a una tarea no exige construir representaciones nuevas, sino **reorientar las que ya existen**: darle más peso a unas, menos a otras, y combinarlas de otra manera para producir la salida deseada.

Sirve la comparación con un médico con experiencia que cambia de hospital: no vuelve a estudiar la carrera, solo aprende los protocolos del nuevo centro. El ajuste fino es una **corrección pequeña** sobre un conocimiento grande, y una corrección pequeña se puede describir con pocos parámetros.

### 1.2. La dimensión intrínseca de una tarea es baja

Esta idea puede medirse. En lugar de entrenar los $D$ parámetros del modelo, se entrena un vector $z$ de solo $d$ dimensiones y se proyecta al espacio completo con una matriz aleatoria fija $P$:

$$
\theta = \theta_0 + P\,z, \qquad z \in \mathbb{R}^{d}, \quad d \ll D
$$

Se llama **dimensión intrínseca** de la tarea al menor $d$ con el que se alcanza, por ejemplo, el 90 % del rendimiento del ajuste fino completo. Aghajanyan et al. (2020) encontraron valores sorprendentemente bajos: con RoBERTa basta entrenar del orden de **200 parámetros** para alcanzar ese 90 % en la tarea de detección de paráfrasis MRPC. Y encontraron también que cuanto mejor preentrenado está el modelo, **menor** es la dimensión intrínseca de las tareas posteriores: el preentrenamiento actúa como un compresor que deja las tareas "a pocos pasos" de distancia.

Es como una mesa de mezclas con millones de controles: para adaptar el sonido a una sala nueva no hay que tocarlos todos, basta con mover unos pocos, siempre que sean los adecuados. Cada método PEFT es, en el fondo, una propuesta distinta sobre **cuáles son esos pocos controles**.

### 1.3. El cambio en los pesos tiene rango bajo

LoRA lleva la idea anterior a cada matriz de pesos. Si el ajuste fino completo transforma $W$ en $W + \Delta W$, la hipótesis es que $\Delta W$ tiene **rango bajo**: aunque sea una matriz de $d \times d$, casi toda su información cabe en unas pocas direcciones.

Una actualización de rango $r$ solo modifica la respuesta de la capa ante $r$ direcciones del espacio de entrada, y deja el resto exactamente como estaba. La interpretación es que la tarea nueva necesita **amplificar unas pocas direcciones** que el modelo ya había aprendido pero a las que daba poca importancia. Los autores de LoRA comprobaron que, en GPT-3, rangos tan pequeños como 1 o 2 ya daban resultados competitivos en algunas tareas.

### 1.4. Se parte exactamente del modelo original

Todos los métodos PEFT se inicializan de forma que, en el primer paso de entrenamiento, **el modelo modificado es idéntico al preentrenado**: en LoRA una de las dos matrices empieza en cero, los *adapters* empiezan siendo (casi) la función identidad gracias a su conexión residual, y los vectores de escalado empiezan en uno. El entrenamiento no parte de una perturbación aleatoria que haya que corregir, sino del mejor punto conocido, y se aleja de él solo lo necesario.

### 1.5. Pocos parámetros, mucha regularización

Con tan poca capacidad entrenable, el modelo **no puede sobrescribir** lo que sabía. Esto actúa como una regularización muy fuerte:

- reduce el sobreajuste cuando hay pocos datos (el caso habitual en dominios específicos);
- reduce el olvido catastrófico: los pesos originales siguen intactos y, de hecho, basta retirar el módulo entrenado para recuperar el modelo de partida.

### En resumen

El ajuste fino es un desplazamiento pequeño y de baja dimensión respecto a un punto de partida muy bueno. Hay tres lugares donde se puede aplicar ese desplazamiento, y de ahí sale la familia de métodos:

- en **la entrada** del modelo (*soft prompts*);
- en **las activaciones** intermedias (*adapters*, (IA)³);
- en **los pesos** (LoRA y sus variantes, BitFit).

## **2. La familia de métodos PEFT**

Los métodos suelen agruparse en tres categorías, según qué parámetros se entrenan:

| Categoría | Idea | Métodos |
|---|---|---|
| **Aditivos** | Se añaden parámetros o módulos nuevos al modelo congelado | Adapters, prompt tuning, prefix tuning, P-tuning, (IA)³ |
| **Selectivos** | Se entrena un subconjunto de los parámetros que ya existen | Últimas capas, BitFit |
| **Por reparametrización** | El cambio en los pesos se expresa con una parametrización de pocos grados de libertad | LoRA, QLoRA, AdaLoRA, DoRA, VeRA |

A ellas se suman los métodos **híbridos**, que combinan varias de las anteriores.

### **2.1. Adapters**

<a href="https://arxiv.org/abs/1902.00751">Houlsby et al. (2019). Parameter-Efficient Transfer Learning for NLP</a>

Un *adapter* es un módulo pequeño y entrenable que se **inserta entre las capas** del Transformer, mientras que todos los pesos originales permanecen congelados. Fue el primer método PEFT de uso generalizado.

<div align="center">
    <img src="./imgs/adapters.webp" width="650px">
</div>

Cada adapter tiene forma de **cuello de botella**: una capa que reduce la dimensión de $d$ a $m$ (con $m \ll d$), una no linealidad, una capa que vuelve a expandir a $d$ y una conexión residual que suma el resultado a la entrada:

$$
h \;\leftarrow\; h + W_{\text{up}}\, f\!\left(W_{\text{down}}\, h\right), \qquad W_{\text{down}} \in \mathbb{R}^{m \times d}, \quad W_{\text{up}} \in \mathbb{R}^{d \times m}
$$

<div align="center">
    <img src="./imgs/adapters.png" width="250px">
</div>

Cada adapter añade $2md + d + m$ parámetros. Con $d = 768$ y $m = 64$ son unos 99.000, frente a los varios millones de la subcapa a la que acompaña. En el diseño original se colocan dos por bloque: uno tras la atención y otro tras la red *feed-forward*.

**Por qué funciona.** El cuello de botella obliga al adapter a quedarse con las $m$ combinaciones de características más relevantes para la tarea y a calcular, a partir de ellas, una **corrección** que se suma a la representación original. Gracias a la conexión residual y a una inicialización cercana a cero, empieza siendo la identidad y solo se desvía de ella donde la tarea lo requiere. Al incluir una no linealidad, puede además calcular características que no son combinación lineal de las existentes.

**Resultados.** En el trabajo original, sobre BERT y el benchmark GLUE, los adapters quedaban a un 0,4 % del ajuste fino completo añadiendo solo un 3,6 % de parámetros por tarea.

**Inconvenientes.** Los adapters son capas adicionales colocadas *en serie*: aumentan la profundidad de la red y, por tanto, la **latencia en inferencia**. Al contener una no linealidad, no pueden fusionarse con los pesos originales.

**Variantes.** La configuración de Pfeiffer utiliza un solo adapter por bloque (tras la red *feed-forward*), con resultados similares y la mitad de parámetros. *AdapterFusion* combina adapters entrenados en distintas tareas. Los *adapters paralelos* se colocan junto a la subcapa en lugar de detrás, lo que los acerca conceptualmente a LoRA.

### **2.2. Soft prompts: prompt tuning, prefix tuning y P-tuning**

Estos métodos no tocan la red: **aprenden la entrada**. Un prompt convencional (*hard prompt*) es texto, y por tanto está limitado a los embeddings de los tokens del vocabulario. Un *soft prompt* es una secuencia de vectores continuos que se optimizan directamente por descenso de gradiente, con el modelo congelado.

**Prompt tuning** (<a href="https://arxiv.org/abs/2104.08691">Lester et al., 2021</a>). Se anteponen $k$ vectores entrenables a los embeddings de la entrada:

$$
\big[\, p_1, \dots, p_k,\; e(x_1), \dots, e(x_n) \,\big], \qquad p_i \in \mathbb{R}^{d}
$$

Los únicos parámetros entrenables son esos $k \cdot d$ valores: con 20 tokens virtuales y $d = 768$, unos 15.000. Es el método más económico. Su rendimiento depende mucho del tamaño del modelo: en modelos pequeños queda por debajo del ajuste fino completo, pero la diferencia se cierra a medida que el modelo crece.

**Prefix tuning** (<a href="https://arxiv.org/abs/2101.00190">Li y Liang, 2021</a>). En lugar de actuar solo en la entrada, se añaden vectores entrenables en **todas las capas**, como claves y valores adicionales del mecanismo de atención. Así, cada capa dispone de unas "posiciones virtuales" a las que todos los tokens pueden atender. Entrenando en torno a un 0,1 % de los parámetros se obtienen resultados comparables al ajuste fino completo en tareas de generación. Para estabilizar el entrenamiento, los prefijos se generan durante el entrenamiento mediante una pequeña red auxiliar que después se descarta.

**P-tuning** (<a href="https://arxiv.org/abs/2103.10385">Liu et al., 2021</a>). Los tokens virtuales se generan con un codificador entrenable (un LSTM o un MLP) y pueden intercalarse en cualquier posición del prompt. Su segunda versión, <a href="https://arxiv.org/abs/2110.07602">P-tuning v2</a>, aplica prompts en todas las capas, de forma muy similar a prefix tuning.

**Por qué funcionan.** Un LLM ya sabe cambiar de comportamiento según su contexto: es lo que hace cuando le damos instrucciones o ejemplos en el prompt. Un soft prompt es **el mejor prompt posible para la tarea**, encontrado por optimización en lugar de por ensayo y error, y sin la restricción de tener que corresponderse con palabras reales: puede situarse en puntos del espacio de embeddings a los que ningún texto llega. No se añade conocimiento al modelo; se le lleva al "estado" en el que resuelve bien la tarea.

**Ventajas e inconvenientes.**

- Con un único modelo congelado se pueden atender muchas tareas, incluso mezcladas en el mismo lote: cada ejemplo lleva su propio prompt.
- Los tokens virtuales ocupan parte de la ventana de contexto y alargan la secuencia.
- La optimización es más delicada que en otros métodos (sensible a la inicialización y a la tasa de aprendizaje), y en modelos pequeños rinden peor.

### **2.3. (IA)³: reescalar activaciones**

<a href="https://arxiv.org/abs/2205.05638">Liu et al. (2022). Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than In-Context Learning</a>

(IA)³ (*Infused Adapter by Inhibiting and Amplifying Inner Activations*) no añade capas ni matrices: aprende tres **vectores de escalado** por bloque, que multiplican elemento a elemento las claves y los valores de la atención y las activaciones intermedias de la red *feed-forward*:

$$
K \leftarrow l_k \odot K, \qquad V \leftarrow l_v \odot V, \qquad h_{\text{ff}} \leftarrow l_{\text{ff}} \odot h_{\text{ff}}
$$

Los vectores se inicializan a uno, de modo que al principio el modelo no cambia. El número de parámetros entrenables es mínimo, del orden del 0,01 % del modelo.

**Por qué funciona.** Es la versión más literal de la idea de reorientar lo que ya existe: no se crean características nuevas, solo se **sube o se baja el volumen** de cada una. Que algo tan simple funcione es una buena prueba de que las características necesarias ya estaban en el modelo. Además, como un escalado puede absorberse en la matriz de pesos contigua, no añade latencia en inferencia.

### **2.4. Métodos selectivos**

No añaden nada al modelo: eligen **qué parámetros existentes se entrenan** y congelan el resto.

- **Entrenar solo las últimas capas** (o solo la cabeza de clasificación). Es el *transfer learning* clásico. Funciona bien cuando la tarea es de clasificación y se parece al preentrenamiento, pero no permite modificar las representaciones de las capas inferiores.
- **BitFit** (<a href="https://arxiv.org/abs/2106.10199">Ben Zaken et al., 2021</a>). Se entrenan únicamente los **términos de sesgo** (*bias*) de la red, que en BERT suponen en torno al 0,1 % de los parámetros. Con conjuntos de datos pequeños o medianos el resultado es competitivo con el ajuste fino completo.

**Por qué funciona BitFit.** El sesgo de una neurona desplaza su umbral de activación. Al cambiar solo los sesgos no se altera *qué* detecta cada neurona, sino *con qué facilidad se activa*: de nuevo, se cambia la importancia relativa de las características existentes sin modificarlas.

Los métodos selectivos no añaden parámetros ni latencia, pero en modelos grandes y tareas de generación suelen quedar por detrás de LoRA.

### **2.5. LoRA: Low-Rank Adaptation**

<a href="https://arxiv.org/abs/2106.09685">Hu et al. (2021). LoRA: Low-Rank Adaptation of Large Language Models</a>

LoRA es hoy el método PEFT más utilizado. En lugar de añadir capas, **reparametriza el cambio en los pesos**: la matriz preentrenada $W$ queda congelada y la actualización $\Delta W$ se expresa como el producto de dos matrices mucho más pequeñas.

<div align="center">
    <img src="./imgs/LoRA.svg" width="550px">
</div>

Para una capa lineal $y = Wx$ con $W \in \mathbb{R}^{d \times d}$:

$$
y = (W + \Delta W)\,x = Wx + \frac{\alpha}{r}\,A\,B\,x, \qquad A \in \mathbb{R}^{d \times r}, \quad B \in \mathbb{R}^{r \times d}, \quad r \ll d
$$

(En el artículo original los nombres de las dos matrices están intercambiados y el producto se escribe $BA$; aquí seguimos la notación de la figura.)

- **Inicialización.** $A$ se inicializa con valores aleatorios y $B$ a cero, de modo que $\Delta W = AB = 0$ al comenzar: el modelo arranca siendo exactamente el preentrenado.
- **Número de parámetros.** Se entrenan $2dr$ valores en lugar de $d^2$. En una matriz de GPT-2 ($d = 768$) con $r = 8$ son 12.288 parámetros frente a 589.824: un 2 %.
- **Hiperparámetros.** El rango $r$ (habitualmente entre 4 y 64) fija la capacidad de la adaptación; $\alpha$ escala su efecto; y hay que decidir a qué matrices se aplica. El artículo original adaptaba solo las proyecciones de consultas y valores de la atención ($W^Q$ y $W^V$); hoy es frecuente aplicarlo a todas las capas lineales.

En una capa de atención, por ejemplo:

$$
Q = (W^Q + \Delta W^Q)\,X, \qquad V = (W^V + \Delta W^V)\,X
$$

**Por qué funciona.** Es la aplicación directa de la hipótesis de rango bajo (apartado 1.3). Puede leerse así: $B$ extrae de la entrada $r$ características y $A$ decide cómo influye cada una en la salida. LoRA es, en realidad, un adapter **lineal y en paralelo** a la matriz original: el mismo cuello de botella, pero sin no linealidad y sumado a $Wx$ en lugar de intercalado detrás.

**La ventaja decisiva: se puede fusionar.** Al ser lineal, una vez terminado el entrenamiento se puede calcular $W' = W + \frac{\alpha}{r}AB$ y sustituir la matriz original. El modelo resultante tiene exactamente la misma arquitectura y **ninguna latencia adicional**. Y si no se fusiona, se pueden mantener muchos módulos LoRA de unos pocos megabytes sobre un único modelo base e intercambiarlos según la tarea. En el artículo original, el punto de control de GPT-3 pasaba de 350 GB a 35 MB por tarea.

#### **Variantes de LoRA**

| Variante | Qué cambia | Intuición |
|---|---|---|
| <a href="https://arxiv.org/abs/2305.14314">**QLoRA**</a> | El modelo base se carga cuantizado a 4 bits y permanece congelado; los módulos LoRA se entrenan en precisión normal sobre él | La cuantización introduce un error pequeño en los pesos, y el módulo LoRA, al entrenarse sobre el modelo ya cuantizado, aprende la tarea compensándolo. Permite ajustar modelos de decenas de miles de millones de parámetros en una sola GPU |
| <a href="https://arxiv.org/abs/2303.10512">**AdaLoRA**</a> | El rango no es el mismo en todas las matrices: se reparte durante el entrenamiento según la importancia de cada una | No todas las capas necesitan cambiar lo mismo; conviene gastar los parámetros donde más aportan |
| <a href="https://arxiv.org/abs/2402.09353">**DoRA**</a> | Separa cada matriz en magnitud y dirección; entrena la magnitud directamente y aplica LoRA a la dirección | El ajuste fino completo modifica magnitud y dirección de forma independiente; LoRA las acopla. Al separarlas se acerca más al ajuste completo, sobre todo con rangos bajos |
| <a href="https://arxiv.org/abs/2310.11454">**VeRA**</a> | $A$ y $B$ son aleatorias, están congeladas y se comparten entre capas; solo se entrenan vectores de escalado | Si bastan unas pocas direcciones, incluso direcciones aleatorias sirven siempre que se aprenda cuánto pesa cada una. Reduce los parámetros otro orden de magnitud |
| <a href="https://arxiv.org/abs/2312.03732">**rsLoRA**</a> | Escala por $\alpha/\sqrt{r}$ en lugar de $\alpha/r$ | Con rangos altos el factor original atenúa demasiado la actualización; este mantiene estable el aprendizaje |

### **2.6. Métodos híbridos y una visión unificada**

Aunque parezcan muy distintos, todos los métodos anteriores hacen lo mismo: **suman una pequeña corrección $\Delta h$ a alguna representación interna $h$ del modelo congelado**,

$$
h \;\leftarrow\; h + \Delta h
$$

y solo difieren en tres decisiones de diseño (<a href="https://arxiv.org/abs/2110.04366">He et al., 2021</a>):

- **qué representación se modifica** (la entrada, la salida de la atención, la de la red *feed-forward*, las claves y valores);
- **cómo se calcula la corrección** (un cuello de botella con o sin no linealidad, un escalado, vectores libres);
- **cómo se inserta** (en serie, detrás de la subcapa, o en paralelo a ella).

Vistos así, prefix tuning resulta ser una forma de adapter en paralelo sobre la atención, y LoRA un adapter lineal en paralelo sobre los pesos. De esta visión nacen los métodos híbridos, que combinan las mejores opciones de cada uno: por ejemplo, el *MAM adapter* (prefix tuning en la atención y adapters paralelos en la red *feed-forward*) o <a href="https://arxiv.org/abs/2110.07577">UniPELT</a>, que incorpora varios métodos a la vez y aprende cuánto activar cada uno.

## **3. Comparativa**

Los porcentajes son órdenes de magnitud orientativos: dependen del modelo y de la configuración.

| Método | Dónde actúa | Parámetros entrenables | Latencia añadida en inferencia | ¿Se fusiona con el modelo? |
|---|---|---|---|---|
| **Adapters** | Activaciones, tras cada subcapa | 1 – 4 % | Sí (capas en serie) | No |
| **Prompt tuning** | Embeddings de entrada | < 0,1 % | Leve (secuencia más larga) | No |
| **Prefix tuning** | Claves y valores de todas las capas | ~ 0,1 % | Leve | No |
| **(IA)³** | Escalado de activaciones | ~ 0,01 % | No | Sí |
| **BitFit** | Sesgos existentes | ~ 0,1 % | No | Son pesos del propio modelo |
| **LoRA** | Matrices de pesos | 0,1 – 1 % | No, una vez fusionado | Sí |

## **4. Consideraciones prácticas**

- **LoRA (o QLoRA, si la memoria es el límite) es la opción por defecto** para ajustar LLMs: buen rendimiento, sin latencia añadida y módulos pequeños e intercambiables.
- **El rango es una medida de cuánto se permite cambiar al modelo.** Rangos de 8 a 16 bastan para tareas cercanas a lo que el modelo ya sabe hacer (cambiar el formato o el tono, seguir una convención). Si la tarea se aleja mucho del preentrenamiento conviene aumentar el rango y adaptar más matrices.
- **PEFT tiene límites.** La hipótesis de baja dimensión se cumple cuando la tarea reorienta conocimiento existente. Cuando hay que incorporar mucho conocimiento nuevo (un idioma que el modelo no domina, un dominio extenso y muy alejado), la distancia respecto al ajuste fino completo aumenta.
- **El ahorro de memoria en entrenamiento es real, pero no proporcional al de parámetros.** Se evitan los gradientes y los estados del optimizador de los pesos congelados, pero la retropropagación sigue atravesando toda la red y hay que almacenar sus activaciones. Por eso se combina a menudo con la cuantización del modelo base (QLoRA).
- **Implementación.** La biblioteca <a href="https://huggingface.co/docs/peft/index">🤗 PEFT</a> de Hugging Face implementa todos los métodos de este notebook con una interfaz común. Los notebooks de la práctica del tema utilizan LoRA a través de ella.

### **Referencias**

- Lialin et al. (2023). [Scaling Down to Scale Up: A Guide to Parameter-Efficient Fine-Tuning](https://arxiv.org/abs/2303.15647)
- Li et al. (2018). [Measuring the Intrinsic Dimension of Objective Landscapes](https://arxiv.org/abs/1804.08838)
- Aghajanyan et al. (2020). [Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning](https://arxiv.org/abs/2012.13255)
- Houlsby et al. (2019). [Parameter-Efficient Transfer Learning for NLP](https://arxiv.org/abs/1902.00751)
- Pfeiffer et al. (2020). [AdapterFusion: Non-Destructive Task Composition for Transfer Learning](https://arxiv.org/abs/2005.00247)
- Lester et al. (2021). [The Power of Scale for Parameter-Efficient Prompt Tuning](https://arxiv.org/abs/2104.08691)
- Li y Liang (2021). [Prefix-Tuning: Optimizing Continuous Prompts for Generation](https://arxiv.org/abs/2101.00190)
- Liu et al. (2021). [GPT Understands, Too](https://arxiv.org/abs/2103.10385) (P-tuning)
- Liu et al. (2022). [Few-Shot Parameter-Efficient Fine-Tuning is Better and Cheaper than In-Context Learning](https://arxiv.org/abs/2205.05638) ((IA)³)
- Ben Zaken et al. (2021). [BitFit: Simple Parameter-efficient Fine-tuning for Transformer-based Masked Language-models](https://arxiv.org/abs/2106.10199)
- Hu et al. (2021). [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- Dettmers et al. (2023). [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)
- He et al. (2021). [Towards a Unified View of Parameter-Efficient Transfer Learning](https://arxiv.org/abs/2110.04366)